In [13]:
from openai import OpenAI
import ollama
from pprint import pprint
import itertools
import json
import pandas as pd

## Playing with parameters

Once you've found a few models or model families that work well, you can also start to play with model parameters.

Let's review a few ones that are usually universal for LLMs:

- [temperature](https://huggingface.co/blog/how-to-generate#sampling): Temperature controls the sampling at the final layer of the LLM in the Softmax layer. Higher temperature (>=1) will result in more tokens being possible (i.e. "more likely to choose a less likely token"). Lower temperature (0-0.5) will result in the most likely token (or few tokens) being sampled.  
- [top-k sampling](https://huggingface.co/blog/how-to-generate#top-k-sampling): Spreading the probability distribution (i.e. chance of being selected) across top k tokens, leading to more likelihood of likely tokens and 0 likelihood of least likely tokens.
- [top-p sampling](https://huggingface.co/blog/how-to-generate#top-p-nucleus-sampling): Less granular than k, top-p chooses whatever set of tokens is equal or greater than p. For example, if p=.9, then it will accumulate as many tokens with their individual probabilities leads to .9 of the overall probability mass of next tokens. Then it spreads the remaining probability (.1) across those tokens, leading to more variety in next token.

There are other ones that you might be interested to look at after playing with these for a bit, like [repetition penalty](https://discuss.huggingface.co/t/transformers-repetition-penalty-parameter/43638) or toggling things like [context window length](https://github.com/ollama/ollama/blob/main/docs/faq.md#how-can-i-specify-the-context-window-size); however, I recommend not overly relying on model parameters to control behavior. Trust your tests and choose a model where you don't need to touch every knob.

Note for math nerds on temperature from the [Softmax wiki page](https://en.wikipedia.org/wiki/Softmax_function) if b > 1 then larger input components will result in larger output probabilities, and increasing the value of b will create probability distributions that are more concentrated around the positions of the largest input values.

You'll need to experiment with these and note that their performance will change with model updates. That means use them sparingly as the main knob for achieving model performance!

In [2]:
model_name = 'gemma3:1b'

In [4]:
prompt_df = pd.read_csv('data/prompts/syntehtic_initial_prompts.csv')

In [5]:
prompt_df

,synthetic_prompt
0,I’m looking for a simple pasta dish. I have sp...
1,"¡Hola! Tengo huevos, tortillas, cebolla y cila..."
2,Looking for something comforting. I have chick...
3,"I want a Mexican dish. I have tortillas, salsa..."
4,"I’m craving something savory. I have potatoes,..."
5,I’m trying to make a dinner for my family. I h...
6,"Hey! I have some chicken breast, potatoes, and..."
7,I want a flavorful vegetarian meal. I have bea...
8,Looking for a quick and easy snack. I have som...
9,"I’m craving a hearty soup. I have potatoes, ce..."


In [6]:
prompt_to_compare = prompt_df.loc[7, 'synthetic_prompt']

In [7]:
prompt_to_compare

'I want a flavorful vegetarian meal. I have beans, corn, and peppers.  Can you give me a recipe that won’t take too long?'

In [8]:
meta_prompt_filename = 'data/meta_prompts/use_case_metaprompt'

In [9]:
def build_conversation(meta_prompt, test_prompt):
    return [
        {'role': 'system',
         'content': meta_prompt},
        {'role': 'user',
         'content': test_prompt}]

def get_response(model_name, conversation, hyperparameters):
    response = ollama.chat(
        model=model_name,
        messages=conversation,
        options=hyperparameters)
    return response.message.content

def prep_and_generate(synthetic_prompt, meta_prompt_filename, model_name, hyperparameters):
    with open(meta_prompt_filename, 'r') as mpf:
        meta_prompt = mpf.read()
    conversation = build_conversation(meta_prompt, synthetic_prompt)
    return get_response(model_name, conversation, hyperparameters)

In [10]:
hyperparameter_options = {
    "temperature": [0.5, 1, 1.5],
    "top_k": [5, 10, 25],
    "top_p": [.5, .7, .9]
}

In [14]:
for t, k, p in itertools.product(*list(hyperparameter_options.values())):
    print("Temperature: {t}, Top-K: {k}, Top-P: {p}".format(t=t, k=k, p=p))
    print(prep_and_generate(prompt_to_compare, meta_prompt_filename, model_name, {"temperature": t, "top_k": k, "top_p": p}))

Temperature: 0.5, Top-K: 5, Top-P: 0.5
Okay, let’s make a vibrant and comforting **Roasted Vegetable & Bean Fiesta Bowl** – inspired by Nigella Lawson’s love of simple, colorful flavors! It’s quick, easy, and packed with goodness.

**Yields:** 2 servings
**Prep time:** 15 minutes
**Cook time:** 25-30 minutes

**Ingredients:**

*   1 tablespoon olive oil
*   1 medium onion, chopped
*   2 cloves garlic, minced
*   1 red bell pepper, chopped
*   1 cup canned black beans, rinsed and drained
*   1 cup canned corn, drained
*   1 teaspoon smoked paprika
*   1/2 teaspoon cumin
*   Salt and pepper to taste
*   1/2 cup cooked quinoa (or rice) – optional, for a heartier base
*   Fresh cilantro, chopped (for garnish)
*   Lime wedges (for serving)

**Instructions:**

1.  **Preheat & Prep:** Preheat your oven to 400°F (200°C).
2.  **Roast the Veggies:**  In a large bowl, toss the chopped onion, bell pepper, and corn with 1 tablespoon of olive oil, salt, and pepper. Spread them in a single layer on a

### Your Turn

- Evaluate as a team a few of the entries at different levels of each variable. Make notes on what you think is best for your use case.